<a href="https://colab.research.google.com/github/mrishab/Semantic-Communication-Paper/blob/main/SemanticCommunication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U sentence-transformers

In [2]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors


# Setting up Data, Embeddings, and the Rebuilder Model

For creating embeddings, I am using existing pretrained model `all-MiniLM-L6-v2` to convert text into 384-dimensional embeddings.

Then I'll use an `MLPRegressor` to learn how to map the embeddings back into the original text.

Defining Sample Sentences

In [3]:
sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Data science is an interdisciplinary field.",
    "Artificial intelligence is transforming the world.",
    "I love working with python and pandas.",
    "Jupyter notebooks are great for prototyping.",
    "Natural language processing allows computers to understand text.",
    "Machine learning models require clean data for training."
]

Encoding the Sentences using pretrained transformer

In [4]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
X = embedder.encode(sentences)

X

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


array([[ 0.04393357,  0.05893442,  0.04817836, ...,  0.05216279,
         0.05610652,  0.10206393],
       [ 0.00783487, -0.0218882 , -0.01355819, ..., -0.01323923,
        -0.02153722, -0.00871548],
       [ 0.03872413, -0.00110554,  0.08271615, ..., -0.02902934,
         0.04854365, -0.03839866],
       ...,
       [-0.10965837, -0.01708993,  0.03392352, ...,  0.05659018,
         0.0610888 ,  0.0243884 ],
       [ 0.02510441,  0.0340961 ,  0.05710059, ...,  0.08061729,
         0.07066565, -0.03364325],
       [ 0.00047341, -0.03664266,  0.09367578, ..., -0.00429332,
        -0.02715967, -0.09085605]], dtype=float32)

Training a Rebuilder model to clean our noisy sentences. Using same sentences as input/output

In [5]:
rebuilder = MLPRegressor(hidden_layer_sizes=(256, 128), max_iter=1000, random_state=42)
rebuilder.fit(X, X)

print("Model trained successfully.")

rebuilder

Model trained successfully.


MLPRegressor(hidden_layer_sizes=(256, 128), max_iter=1000, random_state=42)

### Using K-Nearest Neighbors for Sentence Reconstruction

Instead of generating text character-by-character, we use a `NearestNeighbors` model to map the clean embeddings back to our 'library' of known clean sentences.

In [6]:
knn = NearestNeighbors(n_neighbors=1, metric='cosine')
knn.fit(X)

knn

NearestNeighbors(metric='cosine', n_neighbors=1)

Adding Noise to embeddings in order to simulate communication over a noisy channel like a physical medium.

For generating noise, we're using Gaussian noise so it closely resembles real world.

In [13]:
def test_reconstruction_with_knn(noise_level):
    # 1. Add Gaussian noise to the embeddings
    noise = np.random.normal(0, noise_level, X.shape)
    noisy_X = X + noise

    # 2. Clean noisy data using our MLP Rebuilder
    rebuilt_X = rebuilder.predict(noisy_X)

    # 3. Use KNN to convert our noisy embeddings back into text
    distances, indices = knn.kneighbors(rebuilt_X)

    results = []
    for i, match_idx in enumerate(indices):
        predicted_sentence = sentences[match_idx[0]]

        # Calculate similarities for the table
        noisy_sim = cosine_similarity([X[i]], [noisy_X[i]])[0][0]
        rebuilt_sim = cosine_similarity([X[i]], [rebuilt_X[i]])[0][0]

        results.append({
            "Original Sentence": sentences[i],
            "Rebuilt (KNN) Match": predicted_sentence,
            "Noisy Sim": round(float(noisy_sim), 4),
            "Rebuilt Sim": round(float(rebuilt_sim), 4)
        })

    return pd.DataFrame(results)

# Test the application
display(test_reconstruction_with_knn(noise_level=0.1))

,Original Sentence,Rebuilt (KNN) Match,Noisy Sim,Rebuilt Sim
0,The quick brown fox jumps over the lazy dog.,The quick brown fox jumps over the lazy dog.,0.4442,0.4670
1,Data science is an interdisciplinary field.,Data science is an interdisciplinary field.,0.4736,0.5673
2,Artificial intelligence is transforming the wo...,Artificial intelligence is transforming the wo...,0.4227,0.4771
3,I love working with python and pandas.,I love working with python and pandas.,0.4460,0.6398
4,Jupyter notebooks are great for prototyping.,I love working with python and pandas.,0.4077,0.5353
5,Natural language processing allows computers t...,Natural language processing allows computers t...,0.4441,0.5509
6,Machine learning models require clean data for...,Machine learning models require clean data for...,0.4860,0.6121
